# YOLO26s Detector Training on Roboflow

AdamW + cosine LR, early stopping, periodic checkpoints, and metrics export.

In [1]:
 %pip install ultralytics roboflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations
import json, os, shutil
from datetime import datetime
from pathlib import Path
from typing import Any
from ultralytics import YOLO

In [3]:
MODEL_NAME='yolo26s.pt'  # back to small model — medium OOMs at 1280 on 8GB
EPOCHS=200               # was 100; early stop will still cut short if needed
IMGSZ=1280               # larger images catch small distant signs
BATCH=4                  # reduced to fit 1280px in 8GB VRAM
WORKERS=2                # 2 workers is enough; more just pre-loads more images into RAM
DEVICE=0
OPTIMIZER='AdamW'
LR0=1e-3
LRF=1e-2
WEIGHT_DECAY=5e-4
COS_LR=True
PATIENCE=20              # was 20; 164-class problem needs more time
SAVE_PERIOD=5
MOSAIC=0.0               # disabled: mosaic at 1280px creates 2560x2560 arrays that OOM
DEGREES=5.0              # mild rotation augmentation
OUTPUT_ROOT=Path('runs')/'yolo26s_roboflow'
RUN_NAME=f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DATA_YAML=r'C:\BME\MSC\4. félév\GL\GLHF\datasets\data.yaml'
ROBOFLOW_API_KEY='WUjl0avMvi7XWK2MaB2U'
ROBOFLOW_WORKSPACE='gl-hzi'
ROBOFLOW_PROJECT='plate-detect-8tgbr'
ROBOFLOW_VERSION=1
ROBOFLOW_FORMAT='yolov8'
DATASET_CACHE_DIR=Path.home()/'datasets'/'roboflow'
FORCE_DOWNLOAD=False

In [4]:
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path): return str(obj)
    if hasattr(obj, 'item'):
        try: return obj.item()
        except Exception: return str(obj)
    return obj

def resolve_data_yaml(data_yaml, api_key, workspace, project, version, export_format='yolov8', dataset_cache_dir=Path.home()/'datasets'/'roboflow', force_download=False):
    if data_yaml is not None:
        p = Path(data_yaml).resolve()
        if not p.exists(): raise FileNotFoundError(f'Provided data.yaml not found: {p}')
        return p
    missing=[]
    if not api_key: missing.append('ROBOFLOW_API_KEY')
    if not workspace: missing.append('ROBOFLOW_WORKSPACE')
    if not project: missing.append('ROBOFLOW_PROJECT')
    if version is None: missing.append('ROBOFLOW_VERSION')
    if missing: raise ValueError('Missing dataset config: ' + ', '.join(missing))
    from roboflow import Roboflow
    target_dir=(dataset_cache_dir/workspace/project/f'v{version}').resolve()
    # Check if already downloaded
    if target_dir.exists() and not force_download:
        hits = list(target_dir.rglob('data.yaml'))
        if hits:
            print(f'Using cached dataset: {hits[0]}')
            return hits[0].resolve()
    target_dir.mkdir(parents=True, exist_ok=True)
    rf=Roboflow(api_key=api_key)
    ds=rf.workspace(workspace).project(project).version(version).download(export_format, location=str(target_dir))
    print(f'ds.location: {ds.location}')
    # Search recursively — Roboflow may create a nested subfolder
    for search_root in [Path(ds.location), target_dir]:
        hits = list(search_root.rglob('data.yaml'))
        if hits:
            print(f'Found data.yaml: {hits[0]}')
            return hits[0].resolve()
    raise FileNotFoundError(
        f'data.yaml not found under {ds.location}\n'
        f'Contents: {list(Path(ds.location).rglob("*"))}'
    )

def export_metrics(run_dir: Path, train_result: Any, val_result: Any) -> Path:
    m=run_dir/'metrics'; m.mkdir(parents=True, exist_ok=True)
    summary={
      'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds')+'Z',
      'run_dir': str(run_dir),
      'train_result': to_jsonable(getattr(train_result,'results_dict',{})),
      'val_result': to_jsonable(getattr(val_result,'results_dict',{})),
      'best_checkpoint': str(run_dir/'weights'/'best.pt'),
      'last_checkpoint': str(run_dir/'weights'/'last.pt')
    }
    (m/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    for a in ['results.csv','results.png','args.yaml','confusion_matrix.png']:
        s=run_dir/a
        if s.exists(): shutil.copy2(s, m/s.name)
    return m

In [5]:
data_yaml_path = resolve_data_yaml(
    data_yaml=DATA_YAML,
    api_key=ROBOFLOW_API_KEY,
    workspace=ROBOFLOW_WORKSPACE,
    project=ROBOFLOW_PROJECT,
    version=ROBOFLOW_VERSION,
    export_format=ROBOFLOW_FORMAT,
    dataset_cache_dir=DATASET_CACHE_DIR,
    force_download=FORCE_DOWNLOAD,
)
data_yaml_path

WindowsPath('C:/BME/MSC/4. félév/GL/GLHF/datasets/data.yaml')

In [6]:
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)


Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.11.0+cu128
CUDA build: 12.8


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_NAME)
train_kwargs = {
 'data': str(data_yaml_path), 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH,
 'workers': WORKERS, 'device': DEVICE, 'optimizer': OPTIMIZER,
 'lr0': LR0, 'lrf': LRF, 'weight_decay': WEIGHT_DECAY, 'cos_lr': COS_LR,
 'patience': PATIENCE, 'save': True, 'save_period': SAVE_PERIOD,
 'mosaic': MOSAIC, 'degrees': DEGREES, 'cache': False,
 'project': str(OUTPUT_ROOT.resolve()), 'name': RUN_NAME, 'exist_ok': True,
}
print(json.dumps(to_jsonable(train_kwargs), indent=2))
train_result = model.train(**train_kwargs)
run_dir = Path(getattr(train_result, 'save_dir', OUTPUT_ROOT / RUN_NAME))
print('Run dir:', run_dir)

{
  "data": "C:\\BME\\MSC\\4. f\u00e9l\u00e9v\\GL\\GLHF\\datasets\\data.yaml",
  "epochs": 200,
  "imgsz": 1280,
  "batch": 4,
  "workers": 4,
  "device": 0,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "cos_lr": true,
  "patience": 20,
  "save": true,
  "save_period": 5,
  "mosaic": 0.0,
  "degrees": 5.0,
  "project": "C:\\BME\\MSC\\4. f\u00e9l\u00e9v\\GL\\GLHF\\runs\\yolo26s_roboflow",
  "name": "train_20260517_165538",
  "exist_ok": true
}
Ultralytics 8.4.51  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\BME\MSC\4. flv\GL\GLHF\datasets\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropo

In [ ]:
val_result = model.val(data=str(data_yaml_path), split='val')
metrics_dir = export_metrics(run_dir, train_result, val_result)
print('Metrics folder:', metrics_dir)
print('Best weights:', run_dir / 'weights' / 'best.pt')
print('Last weights:', run_dir / 'weights' / 'last.pt')

Ultralytics 8.4.51  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26s summary (fused): 122 layers, 9,528,648 parameters, 0 gradients, 20.9 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 53.623.0 MB/s, size: 53.0 KB)
val: Scanning C:\BME\MSC\4. félév\GL\GLHF\datasets\valid\labels.cache... 219 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 219/219  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.1it/s 2.8s0.1s
                   all        219       1234      0.476      0.134      0.145      0.104
complementary--accident-area--g3          2          2      0.514        0.5      0.519      0.356
complementary--chevron-left--g2          1          1          0          0          0          0
complementary--chevron-left--g5          3          5      0.505        0.6      0.366      0.204
complementary--chevron-right--g1          1          1          0        

C:\Users\palfi\AppData\Local\Temp\ipykernel_48584\1063670955.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds')+'Z',
